# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Soham334/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why


I am choosing **Lane 2: Refresh / Content Opportunity Scoring**.

The goal is to identify which content pages should be reviewed first for a possible refresh, expansion, protection, pruning, or monitoring action.

I chose this lane because the starter dataset contains observable content and search-performance signals such as impressions, clicks, sessions, CTR, average position, content age, freshness, word count, and engagement metrics. These signals can help identify pages that may deserve attention.

The starter pipeline also provides an initial example of this decision-support problem. It compares a transparent rule-based baseline with machine-learning models and produces a ranked refresh queue with reason codes. This makes the lane suitable for investigating whether a learned ranking can prioritize useful review candidates better than simple fixed rules.

My provisional direction is therefore:

**Use observable page and search-performance signals to rank content pages by how strongly they appear to need human review, while keeping the final decision with a human reviewer.**


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The Question

### Research question

**Which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring based on observable search and engagement signals?**

### Decision

The decision is **which content pages should receive limited review capacity first**.

A review team cannot manually investigate every page equally, so the system should help prioritize the pages where the available evidence suggests that a review may be most useful.

### Action

Based on the ranking, a reviewer could:

* review a page for a potential content refresh;
* investigate a low-CTR page with strong visibility;
* consider expanding a page with meaningful demand but weak performance;
* protect a strong-performing page;
* monitor a page when the evidence is weaker;
* investigate whether an apparent decline is caused by consolidation, seasonality, SERP changes, or noise.

The model would **not automatically publish, rewrite, delete, or change a page**. Its role is decision support: it produces a ranked list that a human can inspect.

### Cost of a wrong recommendation

A false positive could cause limited editorial or SEO resources to be spent reviewing a page that does not actually need attention.

A false negative could be more costly because an important page experiencing a genuine performance problem might be missed.

Therefore, I care particularly about **ranking quality at the top of the queue**, rather than simply maximizing overall classification accuracy. Metrics such as Precision@20, Precision@50, and average precision are appropriate because they correspond to the way the output would actually be used.

The goal is not to claim that a refresh will definitely recover a page. The goal is to identify pages that are worth reviewing first based on available evidence.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [4]:
import os

print("Current folder:")
print(os.getcwd())

print("\nFiles/folders here:")
print(os.listdir())

Current folder:
/content

Files/folders here:
['.config', 'sample_data']


In [5]:
!find /content -type f \( -name "*.csv" -o -name "*.parquet" \) | head -100

/content/sample_data/mnist_test.csv
/content/sample_data/california_housing_train.csv
/content/sample_data/california_housing_test.csv
/content/sample_data/mnist_train_small.csv


In [8]:
!git clone https://github.com/Soham334/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 122, done.
remote: Counting objects: 100% (122/122), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 122 (delta 37), reused 95 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (122/122), 1.84 MiB | 11.17 MiB/s, done.
Resolving deltas: 100% (37/37), done.


In [9]:
%cd /content/flyrank-ml-internship

/content/flyrank-ml-internship


In [10]:
!find . -maxdepth 4 -type f | head -100

./data/raw/content_refresh_anonymized.csv
./GUIDE.md
./notebooks/02_your_first_readable_model.ipynb
./notebooks/01_first_look_and_discovery.ipynb
./notebooks/03_working_with_the_full_release.ipynb
./submission/paper_url.txt
./submission/README.md
./Copy_of_01_first_look_and_discovery.ipynb
./skills/deploying-static-pages/SKILL.md
./skills/framing-ml-problems/SKILL.md
./skills/flyrank/flyrank-context/SKILL.md
./skills/flyrank/flyrank-data/SKILL.md
./skills/training-honest-models/SKILL.md
./skills/writing-data-contracts/SKILL.md
./skills/building-baselines/SKILL.md
./skills/writing-research-papers/SKILL.md
./skills/querying-big-datasets/SKILL.md
./skills/README.md
./skills/auditing-signals/SKILL.md
./skills/hunting-leakage-and-validating/SKILL.md
./skills/directing-your-ai-assistant/SKILL.md
./skills/writing-honest-claims/SKILL.md
./DATA_USE.md
./.github/workflows/personalize.yml
./.github/workflows/smoke-test.yml
./.github/workflows/data-path-smoke.yml
./.git/config
./.git/description
.

In [11]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 30000
Columns: 44


In [12]:
declining = (df["trend_direction"] == "down").sum()
declining_rate = declining / len(df)

correlation = df["search_volume"].corr(df["impressions_90d"])

print("Total rows:", len(df))
print("Total columns:", len(df.columns))
print("Declining pages:", declining)
print(f"Declining rate: {declining_rate:.3%}")
print(f"Search volume vs impressions correlation: {correlation:.3f}")

Total rows: 30000
Total columns: 44
Declining pages: 16262
Declining rate: 54.207%
Search volume vs impressions correlation: 0.001


### What these numbers suggest

The starter dataset contains **30,000 pages across 44 columns**, providing enough observations and multiple observable signals for an initial ranking problem.

There are **16,262 pages labelled as declining**, representing approximately **54.2%** of the starter dataset. This indicates that the starter label is common enough to investigate rather than being an extremely rare event.

One interesting discovery is that the correlation between `search_volume` and `impressions_90d` is only **0.001** in the starter data. This suggests that keyword search volume alone does not explain how much traffic or visibility an individual page actually receives.

This supports the decision to investigate a combination of signals rather than relying on a single metric. The starter pipeline already uses signals such as impressions, clicks, sessions, position, CTR, content age, freshness, and engagement.

These are preliminary observations from the starter dataset, not evidence of causation.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

## What I can claim

I can investigate whether observable content, search, and engagement signals are useful for prioritizing pages for human review.

I can compare a simple baseline with machine-learning approaches and evaluate whether a learned ranking improves the prioritization of pages in the validation data.

I can also investigate which signals are associated with the model's recommendations and use them to provide understandable reason codes for reviewers.

## What I cannot claim

I cannot claim that any individual signal causes a change in Google rankings.

I cannot claim that a page identified as declining will definitely recover after being refreshed.

I cannot claim that the model has discovered a Google ranking factor.

I also should not treat the model's output as an automatic decision to rewrite, publish, or delete a page. The output should support a human reviewer who makes the final decision.

The `trend_direction` field in the starter data is a useful starting label, but it should not automatically be treated as a perfect future outcome. For the later capstone, I would need to carefully define the prediction target and ensure that features do not contain information from the future evaluation period.

Therefore, the current results should be interpreted as evidence that the prioritization problem is worth investigating further, rather than proof that the model will improve SEO outcomes.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

* [x] I selected one of the predefined lanes.
* [x] I explained why I selected the lane.
* [x] I defined the decision the system would support.
* [x] I described an action a human could take from the output.
* [x] I explained the cost of a wrong recommendation.
* [x] I used real numbers from the starter dataset.
* [x] I explained why ML could be useful for this problem.
* [x] I avoided treating the task as simply "train a model."
* [x] I distinguished association from causation.
* [x] I described the model as decision support rather than an automatic decision-maker.
* [x] I identified limitations of the starter label and the need for careful future validation.
